### Enter full names of group members:

##### Name A:
##### Name B:

In [36]:
import math
import numpy as np
from sympy import prime
from pathlib import Path  # for paths of files
import csv
import copy
import random
from sklearn.metrics.pairwise import cosine_similarity

# ANSI escape codes for colors
class colors:
    red = '\033[91m'
    green = '\033[92m'
    blue = '\033[94m'
    end = '\033[0m'  


### 1. DGIM

#### 1.1. DGIM algorithm

In [37]:
# Default DGIM parameters

stream_path = 'data/my_stream.txt'

# The window size
N = 500 

In [38]:
class Bucket:
    def __init__(self, size:int, timestamp):
        self.size = size
        self.timestamp = timestamp

    def __repr__(self):
        return f"Bucket(size={self.size}, timestamp={self.timestamp})"
    
    def get_timestamp(self):
        return self.timestamp
    
    def get_size(self) -> int:
        return self.size
    


def count_sizes(bucket_list: list[Bucket]):
    """
    Count the number of buckets of each size in the bucket list.
    Args:
        bucket_list (list): List of Bucket objects.
    Returns:
        dict: A dictionary where keys are bucket sizes and values are lists of buckets of that size.
    """
    size_counts = {}
    for bucket in bucket_list:
        size = bucket.get_size()
        if size not in size_counts:
            size_counts[size] = []
        size_counts[size].append(bucket)
    return size_counts



def merge_buckets(bucket_list, bucket1:Bucket, bucket2:Bucket):
    """
    Merge two buckets into one by summing their sizes and keeping the timestamp of the older bucket.
    This function modifies the original bucket list in place.
    It removes the two buckets being merged and adds a new bucket with the combined size.
    """
    bucket_list.remove(bucket1)
    bucket_list.remove(bucket2)
    bucket_list.append(Bucket(
        size=bucket1.get_size() + bucket2.get_size(),
        timestamp=bucket1.get_timestamp()
    ))



def dgim_algorithm(stream_path, N):
    
    # Create the buckets list and initialize the timestamp
    bucket_list = []
    timestamp = 0

    # Loop through the entire data stream, one bit at a time
    with open(stream_path) as f:
        
        while True:
            bit = f.read(1)
            if not bit:
                break
            
            # Update the timestamp and remove buckets older than N (outside the current window)
            timestamp += 1 #Update the timestamp, which is the number of bits read so far.
            if len(bucket_list) > 0:
                bucket_list = [bucket for bucket in bucket_list if bucket.get_timestamp() > timestamp - N]
            
            # If the current bit is 1, create a new bucket
            if bit == '1':
                # Create a new bucket as a dictionary
                bucket_list.append(Bucket(1, timestamp))
                
                # Merge buckets if necessary
                need_to_merge = True
                while need_to_merge:
                    need_to_merge = False
                    
                    # Count the sizes of the buckets
                    size_counts = count_sizes(bucket_list)
                    
                    # Check for sizes that need merging (more than 2 buckets of same size)
                    for size in sorted(size_counts.keys()):
                        if len(size_counts[size]) > 2:
                            need_to_merge = True
                            
                            # Sort by timestamp (oldest first)
                            size_counts[size].sort(key=lambda bucket: bucket.get_timestamp())
                
                            # Merge the oldest two buckets of this size
                            merge_buckets(bucket_list, size_counts[size][0], size_counts[size][1])
                            
                            # Break to recalculate sizes
                            break
    
    #Return the final bucket list and the timestamp
    return bucket_list, timestamp

In [39]:
bucket = dgim_algorithm(stream_path, N)

In [40]:
print(f"The updated list of timestamps buckets from DGIM algorithm: \n {bucket[0]}")
print(f"The end timestamp: {bucket[1]}")   

The updated list of timestamps buckets from DGIM algorithm: 
 [Bucket(size=64, timestamp=1009690), Bucket(size=8, timestamp=1010045), Bucket(size=16, timestamp=1010007), Bucket(size=32, timestamp=1009951), Bucket(size=64, timestamp=1009823), Bucket(size=4, timestamp=1010076), Bucket(size=8, timestamp=1010064), Bucket(size=2, timestamp=1010090), Bucket(size=4, timestamp=1010084), Bucket(size=1, timestamp=1010099), Bucket(size=2, timestamp=1010094)]
The end timestamp: 1010102


#### 1.2. Query the Bucket 

In [41]:
def actual_count(stream_path, k):
    stream_list = []
    with open(stream_path, 'r') as file:
        for line in file:
            stream_list.extend(list(map(int, line.strip())))

    # Convert the list into a numpy array
    stream_array = np.array(stream_list)
    
    return int(np.sum(stream_array[-k:]))

In [ ]:
def dgim_query(bucket, N, k):      
    # Extract the buckets and the end timestamp
    bucket_list, end_time_stamp = bucket
   
    # Initialize the different variables
    one_count = 0
    
    # Validate query range
    if k > N:
        raise ValueError("Query range k cannot exceed window size N")
    
    # Calculate the start timestamp for our query window
    query_start = end_time_stamp - k + 1
    
    # Sort buckets by timestamps (newest first)
    sorted_buckets = sorted(bucket_list, key=lambda bucket: bucket.get_timestamp(), reverse=True)
    
    # Process the buckets
    for bucket in sorted_buckets:
        if bucket.get_timestamp() >= query_start:
            # Bucket is entirely within our query window
            one_count += bucket.get_size()
        else:
            # This bucket crosses the boundary (partially in window)
            # We estimate that half of its bits are in our window
            one_count += bucket.get_size() / 2
            break
    
    return math.ceil(one_count)

In [43]:
# List of queries
K = [10, 50, 100, 300, 500] 

In [44]:
print("---------------------------------------------------------------")
for k in K:
    dgim_count = dgim_query(bucket, 500, k)
    true_count = actual_count(stream_path, k)
    
    print(f"The total 1s in the last {k} bits by DGIM: {dgim_count}")
    print(f"The true count of 1s in the last {k} bits: {true_count}")
    print(f"The DGIM error for predicted 1s in the last {k} bits: \
    {round(abs(100*(dgim_count-true_count))/true_count,2)} %")
    print("---------------------------------------------------------------")

---------------------------------------------------------------


TypeError: 'Bucket' object is not subscriptable

### 2. Bloom filters

In [ ]:
# Username data for the creation of bloom filters - B
data_file = (Path("data/bloom_username").with_suffix('.csv'))

# Test data to check the functionality and false positive rate
test1_file = (Path("data/test1_username").with_suffix('.csv'))
test2_file = (Path("data/test2_username").with_suffix('.csv'))

# Default bloom filter parameters
bloom_size = 1500000 # parameter N
h = 3 # number of hash functions

In [ ]:
# create an array of bloom filter with zeros
B = np.zeros(bloom_size)

In [ ]:
B

array([0., 0., 0., ..., 0., 0., 0.], shape=(1500000,))

#### 2.1. Create Bloom filter

In [ ]:
def generate_hash(h, N):
    hash_list = []
    
    # To-do! generate a list of hash functions
        
    return hash_list

In [ ]:
hashes = generate_hash(h, bloom_size)

In [ ]:
def create_bloom_filter(B, hashes, data):
    with data.open() as f:
        for name in f:
            
            # To-do! update the hash index of the bloom filter with 1s
            
    return B

IndentationError: expected an indented block after 'for' statement on line 3 (2174988542.py, line 7)

In [ ]:
bloom_array = create_bloom_filter(B, hashes, data_file)

In [ ]:
bloom_array

array([1., 1., 0., ..., 0., 1., 0.])

#### 2.2. Verify usernames

In [ ]:
def single_verify_username(bloom_array, hashes, new_user):
    
    # To-do! verify username and return a code of 0 or 1 (1 - username taken and 0 - username available)
        
    return code
    

In [ ]:
# Feel free to test different usernames here

new_username = "KazeemTDT4305"

# new_username = "ShambaTDT4305"

In [ ]:
user_code = single_verify_username(bloom_array, hashes, new_username)

In [ ]:
if user_code == 1:
    print(colors.red + f"Username {new_username} has been taken. Try again!" + colors.end)
elif user_code == 0:
    print(colors.green + f"Username {new_username} is available. Congrats!" + colors.end)
else:
    print(colors.blue + f"Wrong pass code. Please reverify!" + colors.end)  

Username KazeemTDT4305 is available. Congrats!


In [ ]:
def group_verify_username(bloom_array, hashes, data):
    # Initialize counts
    total_name = 0
    taken_name = 0
    
    with data.open() as f:
        for name in f:
            # To-do! similar to the single verify, but returns a percentage of usernames taken...
            # ...(In other words seen already by the bloom filter during its creation)
            
    return round(taken_name/total_name*100,2)   

In [ ]:
print("----------------------------------------------------------")
user_total = group_verify_username(bloom_array, hashes, test1_file)
print(f"Percentage of username seen before from test 1: {user_total}%")
print("----------------------------------------------------------")
user_total = group_verify_username(bloom_array, hashes, test2_file)
print(f"Percentage of username seen before from test 2: {user_total}%")
print("----------------------------------------------------------")

----------------------------------------------------------
Percentage of username seen before from test 1: 100.0%
----------------------------------------------------------
Percentage of username seen before from test 2: 23.71%
----------------------------------------------------------


### 3. Flajolet-Martin

In [ ]:
def flajolet_martin(input_stream):
    R = 0  # Initialize maximum rightmost zero bit position to 0

    # To-do! Define hash function h(x) = 6x + 1 mod 5
    

    # To-do! Iterate over the input stream and update maximum rightmost zero bit position
    

    # Estimate the number of distinct elements
    distinct_estimate = 2 ** R

    return distinct_estimate

In [ ]:
# Input stream
input_stream1 = [1, 1, 2, 1, 2, 1, 1, 1, 1, 2, 1, 1]
input_stream2 = [1, 3, 2, 1, 2, 3, 4, 3, 1, 2, 3, 1]

# Run the Flajolet-Martin algorithm
distinct_estimate1 = flajolet_martin(input_stream1)
distinct_estimate2 = flajolet_martin(input_stream2)

# Print the estimated number of distinct elements
print("-----------------------------------------------------")
print(f"Distinct elements (estimated) in input stream 1:", distinct_estimate1)
print("-----------------------------------------------------")
print(f"Distinct elements (estimated) in input stream 2:", distinct_estimate2)
print("-----------------------------------------------------")

-----------------------------------------------------
Distinct elements (estimated) in input stream 1: 2
-----------------------------------------------------
Distinct elements (estimated) in input stream 2: 4
-----------------------------------------------------


### 4. Adword 

#### 4.1. Greedy Algorithm

In [ ]:
# User queries
queries = ["big data", "big data", "big data","bloom filters", "bloom filters", "bloom filters",
           "flajolet martin", "flajolet martin", "flajolet martin", "dgim algorithm", "dgim algorithm", "dgim algorithm"]

In [ ]:
# Company A B C and D keywords and budget $$$
global_companies = {
        'A': ["big data", "bloom filters", 3],
        'B': ["flajolet martin", 3],
        'C': ["flajolet martin", "dgim algorithm", 3],
        'D': ["big data", 3],
    }

In [ ]:
def greedy_algorithm(local_companies, queries):
    # Initial revenue
    revenue = 0
    
    # To-do! update revenue using greedy algorithm
    
    return revenue

In [ ]:
total_revenue = 0
total_trials = 10
print("Starting trials using Greedy Algorithm...")
print("------------------------------------------------")
for i in range(total_trials):
    local_companies = copy.deepcopy(global_companies)
    revenue = greedy_algorithm(local_companies, queries)
    total_revenue = total_revenue + revenue
    print(f"Trial {i+1} - Revenue generated: {revenue}")
print("------------------------------------------------")   
print("Average revenue generated for all trials: ",total_revenue/total_trials)

Starting trials using Greedy Algorithm...
------------------------------------------------
Trial 1 - Revenue generated: 8
Trial 2 - Revenue generated: 9
Trial 3 - Revenue generated: 10
Trial 4 - Revenue generated: 7
Trial 5 - Revenue generated: 9
Trial 6 - Revenue generated: 8
Trial 7 - Revenue generated: 8
Trial 8 - Revenue generated: 9
Trial 9 - Revenue generated: 8
Trial 10 - Revenue generated: 8
------------------------------------------------
Average revenue generated for all trials:  8.4


#### 4.2. Balance Algorithm

In [ ]:
def balance_algorithm(local_companies, queries):
    # Initial revenue
    revenue = 0
    
    # To-do! update revenue using balance algorithm
    
    return revenue

In [ ]:
total_revenue = 0
total_trials = 10
print("Starting trials using Balance Algorithm...")
print("-------------------------------------------")
for i in range(total_trials):
    local_companies = copy.deepcopy(global_companies)
    revenue = balance_algorithm(local_companies, queries)
    total_revenue = total_revenue + revenue
    print(f"Trial {i+1} - Revenue generated: {revenue}")
print("-------------------------------------------")   
print("Average revenue generated for all trials: ",total_revenue/total_trials)

Starting trials using Balance Algorithm...
-------------------------------------------
Trial 1 - Revenue generated: 9
Trial 2 - Revenue generated: 9
Trial 3 - Revenue generated: 10
Trial 4 - Revenue generated: 9
Trial 5 - Revenue generated: 9
Trial 6 - Revenue generated: 8
Trial 7 - Revenue generated: 10
Trial 8 - Revenue generated: 9
Trial 9 - Revenue generated: 9
Trial 10 - Revenue generated: 10
-------------------------------------------
Average revenue generated for all trials:  9.2


### 5. Recommender System

In [ ]:
# Ratings matrix (each row corresponds to a movie, and each column corresponds to a user)
ratings_matrix = np.array([
    [1, 0, 3, 0, 0, 5, 0, 0, 5, 0, 4, 0],
    [0, 0, 5, 4, 0, 0, 4, 0, 0, 2, 1, 3],
    [2, 4, 0, 1, 2, 0, 3, 0, 4, 3, 5, 0],
    [0, 2, 4, 0, 5, 0, 0, 4, 0, 0, 2, 0],
    [0, 0, 4, 3, 4, 2, 0, 0, 0, 0, 2, 5],
    [1, 0, 3, 0, 3, 0, 0, 2, 0, 0, 4, 0]
])

#### 5.1. User-User Collaborative Filtering

In [ ]:
def user_cf(rate_m, tup_mu, neigh):
    
    # To-do! implement a user-user CF using cosine similarity as distance measure
    
    return prediction   

In [ ]:
# List of tuple of movie rating by users to be predicted e.g (1, 5) refers to the rating of movie 1 by user 5
list_mu_query = [(1, 5), (3, 3)]

# Neighbor selection (|N|)
neigh = 2

In [ ]:
print("-----------------------------------------------------------------")   
for mu_query in list_mu_query:
    predicted_rating = user_cf(ratings_matrix, mu_query, neigh)
    print(f"The predicted rating of movie {mu_query[0]} by user {mu_query[1]}: {predicted_rating} (User-User CF)")
    print("-----------------------------------------------------------------")   

-----------------------------------------------------------------
The predicted rating of movie 1 by user 5: 1.42 (User-User CF)
-----------------------------------------------------------------
The predicted rating of movie 3 by user 3: 1.49 (User-User CF)
-----------------------------------------------------------------


#### 5.2. Item-Item Collaborative Filtering

In [ ]:
def item_cf(rate_m, tup_mu, neigh):
    
    # To-do! implement a item-item CF using cosine similarity as distance measure
    
    return prediction

In [ ]:
print("-----------------------------------------------------------------")   
for mu_query in list_mu_query:
    predicted_rating = item_cf(ratings_matrix, mu_query, neigh)
    print(f"The predicted rating of movie {mu_query[0]} by user {mu_query[1]}: {predicted_rating} (Item-Item CF)")
    print("-----------------------------------------------------------------")   

-----------------------------------------------------------------
The predicted rating of movie 1 by user 5: 2.48 (Item-Item CF)
-----------------------------------------------------------------
The predicted rating of movie 3 by user 3: 3.0 (Item-Item CF)
-----------------------------------------------------------------


### Provide concise answers to all 5 cases in the Project 3 description below

#### Case 1

In [ ]:
# Enter answer here

#### Case 2

In [ ]:
# Enter answer here

#### Case 3

In [ ]:
# Enter answer here

#### Case 4

In [ ]:
# Enter answer here

#### Case 5

In [ ]:
# Enter answer here